In [ ]:
# Students: Siddhi Kakani and Simona Matiukaite
# University: Northeastern University, Boston
# Year: 2026
# 
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#CELL 2
import os

# Base directory â€” all project files live under here
BASE = '/kaggle/working/meeting_intelligence'


folders = {
    #  Raw data Downloaded from HuggingFace
    'raw_ami':          f'{BASE}/data/raw/business/ami',
    'raw_ami_summaries':f'{BASE}/data/raw/business/ami_summaries',
    'raw_icsi':         f'{BASE}/data/raw/education/icsi',
    'raw_healthcare':   f'{BASE}/data/raw/healthcare/medical',

    #Processed data (after cleaning & preprocessing) â”€â”€
    'proc_audio_biz':   f'{BASE}/data/processed/audio/business',
    'proc_audio_edu':   f'{BASE}/data/processed/audio/education',
    'proc_audio_health':f'{BASE}/data/processed/audio/healthcare',
    'proc_transcripts': f'{BASE}/data/processed/transcripts',

    #  Final clean dataset (ready for training)
    'final':            f'{BASE}/data/final',
    'final_splits':     f'{BASE}/data/final/splits',

    #Model checkpoint
    'model_bart':       f'{BASE}/models/bart_summarizer',
    'model_bart_best':  f'{BASE}/models/bart_summarizer/best',
    'model_classifier': f'{BASE}/models/domain_classifier',

    # Results & evaluation
    'eval_results':     f'{BASE}/results/evaluation',
    'eval_outputs':     f'{BASE}/results/outputs',
    'eval_samples':     f'{BASE}/results/sample_outputs',

    #Gradio demo assets
    'demo':             f'{BASE}/demo',

    # Logs
    'logs':             f'{BASE}/logs',
}

# Create all folders
for name, path in folders.items():
    os.makedirs(path, exist_ok=True)

In [ ]:
# CELL 3
categories = {
    'Raw Data': ['raw_ami', 'raw_ami_summaries', 'raw_icsi', 'raw_healthcare'],
    'Processed': ['proc_audio_biz', 'proc_audio_edu', 'proc_audio_health', 'proc_transcripts'],
    'Final Dataset': ['final', 'final_splits'],
    'Models': ['model_bart', 'model_bart_best', 'model_classifier'],
    'Results': ['eval_results', 'eval_outputs', 'eval_samples'],
    'Demo & Logs': ['demo', 'logs'],
}

for category, keys in categories.items():
    print(f'  [{category}]')
    for key in keys:
        short_path = folders[key].replace(BASE + '/', '')
        print(f'    â”œâ”€â”€ {short_path}/')
    print()

In [ ]:
!pip install -q transformers==4.40.0 datasets accelerate evaluate
!pip install -q rouge-score nltk sentencepiece
!pip install -q librosa soundfile openai-whisper # Audio processing
!pip install -q tqdm rich
!pip install -q gradio
!pip install -q huggingface_hub




In [ ]:
# CELL 5
import os
import re
import json
import math
import time
import warnings
from datetime import datetime
from pathlib import Path
from collections import defaultdict, Counter


import numpy as np
import pandas as pd

import librosa
import soundfile as sf

import torch
import torch.nn as nn
from torch.utils.data import DataLoader


import transformers
import datasets as hf_datasets
from transformers import (

    BartTokenizer,
    BartForConditionalGeneration,

    # Generic auto classes (for domain classifier)
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
)
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets

# NOTE: We import training utilities (Seq2SeqTrainer, DataCollatorForSeq2Seq,
# etc.) later in Stage 7 right before training. This avoids the peft/
# transformers version conflict at import time on some Kaggle environments.

# â”€â”€ Evaluation â”€â”€
import evaluate
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# â”€â”€ Progress & Display â”€â”€
from tqdm.auto import tqdm

# â”€â”€ Suppress noisy warnings â”€â”€
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import whisper

from IPython.display import Audio #for making a sound
# Generate a simple beep (440 Hz sine wave for 1 second)
sam_ra = 22050  # sample rate
t = np.linspace(0, 1, sam_ra)
tone = np.sin(2 * np.pi * 440 * t)
Audio(tone, rate=sam_ra, autoplay=True)

In [ ]:
from huggingface_hub import login
HF_TOKEN = ""


In [ ]:
# CELL 7
BASE = '/kaggle/working/meeting_intelligence'
DEVICE = 'cuda'

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
print(f'  Base dir : {BASE}')
print(f'  Device   : {DEVICE}')

In [ ]:
#CELL 8
from datasets import load_dataset

print('Loading AMI Meeting Corpus...')
print('This is ~15GB â€” please wait 5-10 minutes...')
print()

ami_dataset = load_dataset(
    "edinburghcstr/ami",
    "ihm",                  # ihm = individual headset microphone (cleaner audio)
    split="train",
    trust_remote_code=True,
)

# Quick inspection
print('AMI Corpus loaded!')
print(f'  Total utterances : {len(ami_dataset):,}')
print(f'  Columns          : {ami_dataset.column_names}')

# How many UNIQUE meetings are in this dataset?
meeting_ids = set(ami_dataset['meeting_id'])
print(f'  Unique meetings  : {len(meeting_ids)}')
print(f'  Sample IDs       : {sorted(list(meeting_ids))[:5]}')

# Show first record
sample = ami_dataset[0]
print(f'\n  Sample record:')
print(f'    meeting_id  : {sample["meeting_id"]}')
print(f'    speaker_id  : {sample["speaker_id"]}')
print(f'    text        : {sample["text"]}')
print(f'    begin_time  : {sample["begin_time"]:.2f}s')
print(f'    end_time    : {sample["end_time"]:.2f}s')
print(f'    duration    : {sample["end_time"] - sample["begin_time"]:.2f}s')
print(f'    sample_rate : {sample["audio"]["sampling_rate"]}Hz')

Audio(tone, rate=sam_ra, autoplay=True)

In [ ]:
from datasets import load_dataset
from huggingface_hub import login

login(token=HF_TOKEN)

ami_summaries = load_dataset("knkarthick/AMI")

print('AMI Summaries loaded!')
print(f'  Train      : {len(ami_summaries["train"])} meetings')
print(f'  Validation : {len(ami_summaries["validation"])} meetings')
print(f'  Test       : {len(ami_summaries["test"])} meetings')

In [ ]:
#CELL 10
total_summaries = (len(ami_summaries["train"])
                   + len(ami_summaries["validation"])
                   + len(ami_summaries["test"]))
print(f'  TOTAL      : {total_summaries} meetings with summaries')
print(f'  Columns    : {ami_summaries["train"].column_names}')

# Show a sample â€” this is what our BART model will learn from
sample_sum = ami_summaries["train"][0]
print(f'\n  Sample meeting:')
print(f'    ID      : {sample_sum["id"]}')
print(f'    Dialogue: {sample_sum["dialogue"][:200]}...')
print(f'    Summary : {sample_sum["summary"][:200]}...')
print(f'\n    Dialogue length : {len(sample_sum["dialogue"].split())} words')
print(f'    Summary length  : {len(sample_sum["summary"].split())} words')

In [ ]:
# ================================================================
# CELL 12: LOAD HEALTHCARE DATA
# ================================================================

print('Loading healthcare dataset...')

health_dataset = load_dataset("medalpaca/medical_meadow_medqa", split="train")

print(f'  Loaded: Medical Meadow MedQA')
print(f'  Records: {len(health_dataset):,}')
print(f'  Columns: {health_dataset.column_names}')

sample_h = health_dataset[0]
for key in sample_h:
    print(f'    {key}: {str(sample_h[key])[:100]}...')
Audio(tone, rate=sam_ra, autoplay=True)
    

In [ ]:
#CELL 13
from collections import defaultdict

print('Analyzing meeting durations...')

meeting_stats = defaultdict(lambda: {
    'min_time': float('inf'),
    'max_time': 0,
    'num_utterances': 0,
    'speakers': set(),
})

for i in tqdm(range(len(ami_dataset)), desc='Scanning utterances'):
    row = ami_dataset[i]
    mid = row['meeting_id']
    meeting_stats[mid]['min_time'] = min(meeting_stats[mid]['min_time'], row['begin_time'])
    meeting_stats[mid]['max_time'] = max(meeting_stats[mid]['max_time'], row['end_time'])
    meeting_stats[mid]['num_utterances'] += 1
    meeting_stats[mid]['speakers'].add(row['speaker_id'])

durations = []
for mid, stats in meeting_stats.items():
    duration_min = (stats['max_time'] - stats['min_time']) / 60.0
    durations.append({
        'meeting_id': mid,
        'duration_min': round(duration_min, 1),
        'num_utterances': stats['num_utterances'],
        'num_speakers': len(stats['speakers']),
    })

durations = sorted(durations, key=lambda x: x['duration_min'], reverse=True)
df_dur = pd.DataFrame(durations)

print(f'\nTotal unique meetings: {len(durations)}')
print(f'\nDURATION BREAKDOWN:')
print(f'  60+ min  : {len(df_dur[df_dur["duration_min"] >= 60])} meetings')
print(f'  30-60 min: {len(df_dur[(df_dur["duration_min"] >= 30) & (df_dur["duration_min"] < 60)])} meetings')
print(f'  20-30 min: {len(df_dur[(df_dur["duration_min"] >= 20) & (df_dur["duration_min"] < 30)])} meetings')
print(f'  10-20 min: {len(df_dur[(df_dur["duration_min"] >= 10) & (df_dur["duration_min"] < 20)])} meetings')
print(f'  < 10 min : {len(df_dur[df_dur["duration_min"] < 10])} meetings')

print(f'\nTOP 15 LONGEST MEETINGS:')
print('-' * 55)
print(f'{"Meeting ID":<15} {"Duration":<12} {"Utterances":<12} {"Speakers"}')
print('-' * 55)
for _, row in df_dur.head(15).iterrows():
    print(f'{row["meeting_id"]:<15} {row["duration_min"]:>6.1f} min   {row["num_utterances"]:>6}       {row["num_speakers"]}')
print('-' * 55)

good = df_dur[df_dur['duration_min'] >= 30]
okay = df_dur[df_dur['duration_min'] >= 20]
print(f'\nMeetings >= 30 min: {len(good)}')
print(f'Meetings >= 20 min: {len(okay)}')

df_dur.to_csv(f'{BASE}/data/raw/business/ami_meeting_durations.csv', index=False)
print('\nStage 1 COMPLETE!')

Audio(tone, rate=sam_ra, autoplay=True)


In [ ]:
#CELL 14
#Filter meetings >= 30 min with enough content
qualified = df_dur[
    (df_dur['duration_min'] >= 30) &
    (df_dur['num_speakers'] >= 3) &
    (df_dur['num_utterances'] >= 400)
].copy()

print(f'Qualified meetings (30+ min, 3+ speakers, 400+ utterances): {len(qualified)}')
print()

# Split by meeting type based on ID prefix
# ES/IS/TS = scenario-based (design team meetings) â†’ business
# EN/IN = naturally occurring meetings â†’ education/research
qualified['prefix'] = qualified['meeting_id'].str[:2]

scenario_ids = qualified[qualified['prefix'].isin(['ES', 'IS', 'TS'])]['meeting_id'].tolist()
natural_ids = qualified[qualified['prefix'].isin(['EN', 'IN'])]['meeting_id'].tolist()

print(f'Scenario meetings (business candidates): {len(scenario_ids)}')
print(f'Natural meetings (education candidates): {len(natural_ids)}')

# Select top 15 business (longest scenario meetings)
business_meetings = qualified[
    qualified['meeting_id'].isin(scenario_ids)
].sort_values('duration_min', ascending=False).head(15)['meeting_id'].tolist()

# Select top 10 education (longest natural meetings)
education_ami_meetings = qualified[
    qualified['meeting_id'].isin(natural_ids)
].sort_values('duration_min', ascending=False).head(10)['meeting_id'].tolist()

print(f'\nSelected {len(business_meetings)} business meetings:')
for mid in business_meetings:
    row = df_dur[df_dur['meeting_id'] == mid].iloc[0]
    print(f'  {mid:<12} {row["duration_min"]:>5.1f} min  {row["num_speakers"]} speakers  {row["num_utterances"]} utterances')

print(f'\nSelected {len(education_ami_meetings)} education meetings:')
for mid in education_ami_meetings:
    row = df_dur[df_dur['meeting_id'] == mid].iloc[0]
    print(f'  {mid:<12} {row["duration_min"]:>5.1f} min  {row["num_speakers"]} speakers  {row["num_utterances"]} utterances')

Audio(tone, rate=sam_ra, autoplay=True)


In [ ]:
# CELL 15: SELECT BEST MEETINGS
qualified = df_dur[
    (df_dur['duration_min'] >= 30) &
    (df_dur['num_speakers'] >= 3) &
    (df_dur['num_utterances'] >= 400)
].copy()

qualified['prefix'] = qualified['meeting_id'].str[:2]

scenario_ids = qualified[qualified['prefix'].isin(['ES', 'IS', 'TS'])]['meeting_id'].tolist()
natural_ids = qualified[qualified['prefix'].isin(['EN', 'IN'])]['meeting_id'].tolist()

business_meetings = qualified[
    qualified['meeting_id'].isin(scenario_ids)
].sort_values('duration_min', ascending=False).head(15)['meeting_id'].tolist()

education_ami_meetings = qualified[
    qualified['meeting_id'].isin(natural_ids)
].sort_values('duration_min', ascending=False).head(10)['meeting_id'].tolist()

print(f'Business meetings selected: {len(business_meetings)}')
for mid in business_meetings:
    r = df_dur[df_dur['meeting_id'] == mid].iloc[0]
    print(f'  {mid:<12} {r["duration_min"]:>5.1f} min')

print(f'\nEducation meetings selected: {len(education_ami_meetings)}')
for mid in education_ami_meetings:
    r = df_dur[df_dur['meeting_id'] == mid].iloc[0]
    print(f'  {mid:<12} {r["duration_min"]:>5.1f} min')

Audio(tone, rate=sam_ra, autoplay=True)


In [ ]:
# CELL 16: RECONSTRUCT FULL TRANSCRIPTS
def reconstruct_meeting_text(dataset, meeting_id):
    utterances = []
    for i in range(len(dataset)):
        row = dataset[i]
        if row['meeting_id'] == meeting_id:
            utterances.append({
                'text': row['text'],
                'speaker': row['speaker_id'],
                'begin': row['begin_time'],
                'end': row['end_time'],
            })
    if not utterances:
        return None

    utterances.sort(key=lambda x: x['begin'])

    transcript_lines = []
    for utt in utterances:
        transcript_lines.append(f"[{utt['speaker']}]: {utt['text']}")

    full_transcript = '\n'.join(transcript_lines)
    speakers = list(set(u['speaker'] for u in utterances))
    duration_sec = utterances[-1]['end'] - utterances[0]['begin']

    return {
        'meeting_id': meeting_id,
        'transcript': full_transcript,
        'duration_sec': round(duration_sec, 2),
        'duration_min': round(duration_sec / 60, 1),
        'num_utterances': len(utterances),
        'num_speakers': len(speakers),
        'speakers': speakers,
        'word_count': len(full_transcript.split()),
    }

# Business
print('Reconstructing business meetings...')
business_records = []
for i, mid in enumerate(business_meetings):
    print(f'  [{i+1}/{len(business_meetings)}] {mid}...', end=' ')
    record = reconstruct_meeting_text(ami_dataset, mid)
    if record:
        record['domain'] = 'business'
        business_records.append(record)
        print(f'{record["duration_min"]} min, {record["word_count"]} words')
    else:
        print('SKIPPED')

# Education
print('\nReconstructing education meetings...')
education_records = []
for i, mid in enumerate(education_ami_meetings):
    print(f'  [{i+1}/{len(education_ami_meetings)}] {mid}...', end=' ')
    record = reconstruct_meeting_text(ami_dataset, mid)
    if record:
        record['domain'] = 'education'
        education_records.append(record)
        print(f'{record["duration_min"]} min, {record["word_count"]} words')
    else:
        print('SKIPPED')

print(f'\nDone! Business: {len(business_records)}, Education: {len(education_records)}')

Audio(tone, rate=sam_ra, autoplay=True)

In [ ]:
# CELL 17: BUILD HEALTHCARE MEETINGS
health_dataset = load_dataset("medalpaca/medical_meadow_medqa", split="train")

healthcare_records = []
for meeting_idx in range(15):
    start = meeting_idx * 25
    end = start + 25
    if end > len(health_dataset):
        break

    lines = []
    for i in range(start, end):
        sample = health_dataset[i]
        if sample.get('input', ''):
            lines.append(f'[Patient]: {sample["input"]}')
        if sample.get('output', ''):
            lines.append(f'[Doctor]: {sample["output"]}')

    transcript = '\n'.join(lines)
    healthcare_records.append({
        'meeting_id': f'HEALTH_{meeting_idx+1:03d}',
        'domain': 'healthcare',
        'transcript': transcript,
        'duration_sec': None,
        'duration_min': None,
        'num_utterances': len(lines),
        'num_speakers': 2,
        'speakers': ['Doctor', 'Patient'],
        'word_count': len(transcript.split()),
    })

print(f'Healthcare meetings built: {len(healthcare_records)}')
print(f'Avg words: {np.mean([r["word_count"] for r in healthcare_records]):.0f}')

Audio(tone, rate=sam_ra, autoplay=True)

In [ ]:
# CELL 18: MATCH SUMMARIES TO MEETINGS
all_summary_list = []
for split_name in ['train', 'validation', 'test']:
    for item in ami_summaries[split_name]:
        all_summary_list.append(item['summary'])

print(f'Total summaries available: {len(all_summary_list)}')

idx = 0
for record in business_records + education_records:
    if idx < len(all_summary_list):
        record['reference_summary'] = all_summary_list[idx]
        record['has_summary'] = True
        idx += 1
    else:
        record['reference_summary'] = None
        record['has_summary'] = False

for record in healthcare_records:
    record['reference_summary'] = None
    record['has_summary'] = False

print(f'Meetings with summaries: {idx}')

In [ ]:
# CELL 19: MERGE ALL DOMAINS
all_meetings = business_records + education_records + healthcare_records

print('=' * 60)
print('  FINAL MEETING COLLECTION')
print('=' * 60)
print(f'  Business    : {len(business_records)}')
print(f'  Education   : {len(education_records)}')
print(f'  Healthcare  : {len(healthcare_records)}')
print(f'  TOTAL       : {len(all_meetings)}')
print()

for domain in ['business', 'education', 'healthcare']:
    recs = [r for r in all_meetings if r['domain'] == domain]
    words = [r['word_count'] for r in recs]
    has_sum = sum(1 for r in recs if r.get('has_summary', False))
    print(f'  [{domain.upper()}]')
    print(f'    Meetings   : {len(recs)}')
    print(f'    Avg words  : {np.mean(words):.0f}')
    print(f'    Has summary: {has_sum}/{len(recs)}')
    print()

Audio(tone, rate=sam_ra, autoplay=True)

In [ ]:
# CELL 20: SAVE EVERYTHING
import json

FINAL_DIR = f'{BASE}/data/final'

transcript_records = []
for record in all_meetings:
    transcript_records.append({
        'meeting_id': record['meeting_id'],
        'domain': record['domain'],
        'transcript': record['transcript'],
        'duration_min': record.get('duration_min'),
        'num_utterances': record['num_utterances'],
        'num_speakers': record['num_speakers'],
        'speakers': record['speakers'],
        'word_count': record['word_count'],
        'has_summary': record.get('has_summary', False),
        'reference_summary': record.get('reference_summary'),
    })

with open(f'{FINAL_DIR}/all_meetings.json', 'w') as f:
    json.dump(transcript_records, f, indent=2)

csv_data = [{k: v for k, v in r.items() if k not in ['transcript', 'reference_summary']}
            for r in transcript_records]
pd.DataFrame(csv_data).to_csv(f'{FINAL_DIR}/all_meetings_summary.csv', index=False)

print('Files saved!')
print(f'  JSON: {FINAL_DIR}/all_meetings.json')
print(f'  CSV : {FINAL_DIR}/all_meetings_summary.csv')
print()
print('STAGE 2 COMPLETE!')

Audio(tone, rate=sam_ra, autoplay=True)

In [ ]:
# â”€â”€ Stage 3: Audio Preprocessing â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SAMPLE_MEETINGS = business_meetings[:1] + education_ami_meetings[:1]
print(f'Processing {len(SAMPLE_MEETINGS)} sample meetings: {SAMPLE_MEETINGS}')

def combine_utterances(meeting_ids):
    all_m = {}
    for mid in meeting_ids:
        print(f'  Extracting {mid}...', end='')
        utts = []
        for i in range(len(ami_dataset)):
            s = ami_dataset[i]
            if s['meeting_id'] == mid:
                try: utts.append({'audio': s['audio']['array'], 'sampling_rate': s['audio']['sampling_rate'], 'begin': s['begin_time'], 'end': s['end_time'], 'speaker': s['speaker_id']})
                except: pass
        utts.sort(key=lambda x: x['begin'])
        all_m[mid] = utts
        print(f' {len(utts)} utterances')
    return all_m

sample_utterances = combine_utterances(SAMPLE_MEETINGS)

def concatenate_meeting_audio(d):
    out = {}
    for mid, utts in d.items():
        print(f'  Concatenating {mid}...', end=' ')
        combined = np.concatenate([u['audio'] for u in utts])
        out[mid] = {'audio': combined, 'sample_rate': utts[0]['sampling_rate'], 'duration_sec': len(combined)/utts[0]['sampling_rate']}
        print(f'{out[mid]["duration_sec"]/60:.1f} min')
    return out

sample_full = concatenate_meeting_audio(sample_utterances)

def resample_normalize(d, target_sr=16000):
    out = {}
    for mid, data in d.items():
        audio = data['audio']; sr = data['sample_rate']
        if sr != target_sr: audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        mx = np.max(np.abs(audio))
        if mx > 0: audio = audio / mx
        out[mid] = {'audio': audio, 'sample_rate': target_sr, 'duration_sec': len(audio)/target_sr}
        print(f'  {mid}: {out[mid]["duration_sec"]/60:.1f} min')
    return out

sample_processed = resample_normalize(sample_full)

def chunk_audio(d, chunk_sec=30, overlap_sec=2):
    out = {}
    for mid, data in d.items():
        audio = data['audio']; sr = data['sample_rate']
        step = sr * (chunk_sec - overlap_sec); cs = sr * chunk_sec
        chunks = [audio[i:i+cs] for i in range(0, len(audio), step)]
        out[mid] = {'chunks': chunks, 'sample_rate': sr, 'num_chunks': len(chunks)}
        print(f'  {mid}: {len(chunks)} chunks')
    return out

sample_chunks = chunk_audio(sample_processed)

def save_chunks(d, folder):
    for mid, data in d.items():
        p = f'{folder}/{mid}'; os.makedirs(p, exist_ok=True)
        for i, ch in enumerate(data['chunks']): sf.write(f'{p}/chunk_{i:04d}.wav', ch, data['sample_rate'])
        print(f'  {mid}: {data["num_chunks"]} wav files saved')

save_chunks(sample_chunks, f'{BASE}/data/processed/audio')

DEMO_AUDIO_PATH = f'{BASE}/data/demo_meeting.wav'
demo_mid = SAMPLE_MEETINGS[0]
sf.write(DEMO_AUDIO_PATH, sample_processed[demo_mid]['audio'], sample_processed[demo_mid]['sample_rate'])
print(f'\nDemo audio: {DEMO_AUDIO_PATH} ({sample_processed[demo_mid]["duration_sec"]/60:.1f} min)')
print('Stage 3 complete!')

In [ ]:
import gc

# Test audio
sample_rate = 16000; duration = 10
t = np.linspace(0, duration, sample_rate * duration)
test_audio = (0.3*np.sin(2*np.pi*200*t) + 0.2*np.sin(2*np.pi*400*t) + 0.1*np.random.randn(len(t))).astype(np.float32)
test_audio = test_audio / np.max(np.abs(test_audio))

DEMO_AUDIO_PATH = f'{BASE}/data/demo_test.wav'
os.makedirs(f'{BASE}/data', exist_ok=True)
sf.write(DEMO_AUDIO_PATH, test_audio, sample_rate)
print(f'Test audio: {DEMO_AUDIO_PATH}')

print('Loading Whisper small...')
whisper_model = whisper.load_model("small", device="cpu")
result = whisper_model.transcribe(DEMO_AUDIO_PATH)
print(f'Whisper output: "{result["text"][:100]}"')
print('Whisper pipeline WORKS!')

whisper_transcripts = {}
if all_meetings:
    whisper_transcripts[all_meetings[0]['meeting_id']] = all_meetings[0]['transcript']
    print(f'Demo transcript: {all_meetings[0]["meeting_id"]}')

del whisper_model; gc.collect(); torch.cuda.empty_cache()
print('Whisper freed. Stage 3-4 done!')

In [ ]:
# CELL 30
import re

def clean_transcript(text):
    """Clean a meeting transcript for model input."""
    if not text:
        return ''

    # Remove filler words
    fillers = [r'\bum\b', r'\buh\b', r'\bhmm\b', r'\berm\b',
               r'\bah\b', r'\bmm\b', r'\bmhm\b', r'\bhuh\b']
    for f in fillers:
        text = re.sub(f, '', text, flags=re.IGNORECASE)

    # Fix multiple spaces
    text = re.sub(r'  +', ' ', text)

    # Fix space before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    # Remove empty speaker lines like "[SPK]: "
    text = re.sub(r'\[.*?\]:\s*\n', '', text)

    # Remove extra blank lines
    text = re.sub(r'\n\s*\n', '\n', text)

    # Strip whitespace
    text = text.strip()

    return text


# Clean all 40 evaluation meetings
print('Cleaning 40 evaluation meeting transcripts...')
for record in all_meetings:
    original_len = len(record['transcript'].split())
    record['transcript'] = clean_transcript(record['transcript'])
    new_len = len(record['transcript'].split())
    record['word_count'] = new_len

# Show cleaning results
for domain in ['business', 'education', 'healthcare']:
    recs = [r for r in all_meetings if r['domain'] == domain]
    avg_words = np.mean([r['word_count'] for r in recs])
    print(f'  {domain:<12}: avg {avg_words:.0f} words after cleaning')

print(f'\nCleaned {len(all_meetings)} meetings')

Audio(tone, rate=sam_ra, autoplay=True)

In [ ]:
# CELL 31
def clean_dialogue(text):
    """Clean a dialogue from knkarthick/AMI for BART input."""
    if not text:
        return ''
    # Remove filler words
    fillers = [r'\bum\b', r'\buh\b', r'\bhmm\b', r'\berm\b',
               r'\bah\b', r'\bmm\b', r'\bmhm\b']
    for f in fillers:
        text = re.sub(f, '', text, flags=re.IGNORECASE)
    # Fix spaces
    text = re.sub(r'  +', ' ', text)
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = text.strip()
    return text


print('Preparing BART training data from AMI summaries...')
print(f'  Source: knkarthick/AMI ({len(ami_summaries["train"])} train, {len(ami_summaries["validation"])} val, {len(ami_summaries["test"])} test)')
print()

# Process each split
train_data = []
for item in ami_summaries['train']:
    dialogue = clean_dialogue(item['dialogue'])
    summary = item['summary'].strip()
    if dialogue and summary and len(dialogue.split()) > 50:
        train_data.append({
            'dialogue': dialogue,
            'summary': summary,
            'dialogue_words': len(dialogue.split()),
            'summary_words': len(summary.split()),
        })

val_data = []
for item in ami_summaries['validation']:
    dialogue = clean_dialogue(item['dialogue'])
    summary = item['summary'].strip()
    if dialogue and summary and len(dialogue.split()) > 50:
        val_data.append({
            'dialogue': dialogue,
            'summary': summary,
            'dialogue_words': len(dialogue.split()),
            'summary_words': len(summary.split()),
        })

test_data = []
for item in ami_summaries['test']:
    dialogue = clean_dialogue(item['dialogue'])
    summary = item['summary'].strip()
    if dialogue and summary and len(dialogue.split()) > 50:
        test_data.append({
            'dialogue': dialogue,
            'summary': summary,
            'dialogue_words': len(dialogue.split()),
            'summary_words': len(summary.split()),
        })

print(f'  After cleaning:')
print(f'    Train : {len(train_data)} meetings')
print(f'    Val   : {len(val_data)} meetings')
print(f'    Test  : {len(test_data)} meetings')
print(f'    Total : {len(train_data) + len(val_data) + len(test_data)} meetings')

# Show stats
print(f'\n  Training data stats:')
print(f'    Avg dialogue length : {np.mean([d["dialogue_words"] for d in train_data]):.0f} words')
print(f'    Avg summary length  : {np.mean([d["summary_words"] for d in train_data]):.0f} words')
print(f'    Max dialogue length : {max([d["dialogue_words"] for d in train_data])} words')
print(f'    Min dialogue length : {min([d["dialogue_words"] for d in train_data])} words')

Audio(tone, rate=sam_ra, autoplay=True)

In [ ]:
#CELL 32
#Stage 6: EDA statistics: words counts, durations, domain charts
import matplotlib.pyplot as plt

#Step 1: word count statistics per domain. Calculate the average, min and max word counts for business, education and healthcare transcripts. 
#This shows how different the 3 domains are in terms of meeting length.
print('Word count statistics per domain:')
domains = ['business', 'education', 'healthcare']

for domain in domains:
    records = []
    
    for i in all_meetings:
        if i['domain'] == domain:
            records.append(i)

    words = []
    for j in records:
        words.append(j['word_count'])

    print(f'{domain.upper()}:')
    print(f'Total number of meetings: {len(records)}.')
    print(f'Average meeting words: {np.mean(words):.0f}.')
    print(f'Min words: {min(words)}.')
    print(f'Max words: {max(words)}.\n')

#Step 2: duration statistics. Same idea but for meeting duration in minutes. Healthcare won't have this since those are synthetic, so you'd handle just business and education.
print('\nDuration statistics per domain (excluding the healthcare):')

domains = ['business', 'education']

for domain in domains:
    records = []

    for i in all_meetings:
        if i['domain'] == domain and i['duration_min'] is not None:
            records.append(i)

    durations = []
    for j in records:
        durations.append(j['duration_min'])

    print(f'{domain.upper()}:')
    print(f'Total number of meetings: {len(records)}')
    print(f'Average meeting time: {np.mean(durations):.1f} min.')
    print(f'Min meeting time: {min(durations)} min.')
    print(f'Max meeting time: {max(durations)} min.\n')

#Step 3: speaker count analysis. How many speakers per meeting in each domain? Business and education have 3-5, healthcare always has 2.
print('\nSpeaker count analysis per domain:')

domains = ['business', 'education', 'healthcare']
for domain in domains:
    records = []

    for i in all_meetings:
        if i['domain'] == domain:
            records.append(r)

    speakers = []
    for j in records:
        speakers.append(r['num_speakers'])

    print(f'{domain.upper()}:')
    print(f'Total number of meetings: {len(records)}')
    print(f'Average number of speakers: {np.mean(speakers):.1f}')
    print(f'Min number of speakers: {min(speakers)}')
    print(f'Max number of speakers: {max(speakers)}\n')
    

#Step 4: summary length statistics. For the 25 meetings that have reference summaries, look at how long the summaries are compared to the dialogues.
#What's the compression ratio (dialogue words divided by summary words)?
print('Summary lenght statistics:')

#Extract the meetings with summaries only.
meetings_w_summaries = []
for i in all_meetings:
    if i['has_summary'] == True:
        meetings_w_summaries.append(i)

#The length and compression rates of all summaries
summaries_length = []
compression_ratios = []

for j in meetings_w_summaries:
    dialogues_per_summary = j['word_count']
    words_per_summary = len(j['reference_summary'].split())
    ratio = dialogues_per_summary / words_per_summary

    summaries_length.append(words_per_summary)
    compression_ratios.append(ratio)

print(f'Number or meetings with summaries: {len(meetings_w_summaries)}')
print(f'Summary lengths:')
print(f'Average words in a summary: {np.mean(summaries_length):.0f}.')
print(f'Min words in a summary: {min(summaries_length)}.')
print(f'Max words in a summary: {max(summaries_length)}.')

print('Compression ration:')
print(f'Average ratio: {np.mean(compression_ratios):.1f}x')
print(f'Min: {min(compression_ratios):.1f}x')
print(f'Max: {max(compression_ratios):.1f}x')
print(f'BART model will need to compress approximatelly {np.mean(compression_ratios):.0f} words into 1 word.')

#Step 5: visualizations (graphs).
#Chart 1: average word count by domain
average_words = []

for domain in domains:
    words = []
    for i in all_meetings:
        if i['domain'] == domain:
            words.append(i['word_count'])
    average_words.append(np.mean(words))

plt.figure(figsize = (6, 5))
domains_display = ['Business', 'Education', 'Healthcare']
bars = plt.bar(domains_display, average_words, width = 0.5, color= ['#a8d8ea', '#3a7ca5', '#1a3a5c'])

for bar, value in zip(bars, average_words):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100, f'{value:.0f}', ha ='center', fontsize=12)

plt.title(f'Average word count by domain', fontsize=14)
plt.xlabel('Domain')
plt.ylabel('Average word count')
plt.ylim(0, 18000)
plt.xlim(-0.4, 2.4)
plt.tight_layout()
plt.savefig(f'{BASE}/results/evaluation/chart1_word_counts.png')
plt.show()

#Chart 2: Average speakers per domain
average_speakers = []
for domain in domains:
    speakers = []
    for i in all_meetings:
        if i['domain'] == domain:
            speakers.append(i['num_speakers'])
    average_speakers.append(np.mean(speakers))

plt.figure(figsize= (6, 5))
bars = plt.bar(domains, average_speakers, width = 0.5, color=['#2196F3', '#9C27B0', '#4CAF50'])

for bar, value in zip(bars, average_speakers):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, f'{value:.1f}', ha = 'center', fontsize=12)

plt.title('Average Number of Speakers by Domain', fontsize=14)
plt.xlabel('Domain')
plt.ylabel('Average speaker count')
plt.ylim(0, 10)
plt.xlim(-0.4, 2.4)
plt.tight_layout()
plt.savefig(f'{BASE}/results/evaluation/chart3_speakers.png')
plt.show()

#Chart 3: Dialog length vs summary length (we are not including healthcare as there was no audio meetings found).
summary_words = []
dialogue_words = []
domain_labels = []

for i in all_meetings:
    if i['has_summary'] == True:
        dialogue_words.append(i['word_count'])
        summary_words.append(len(i['reference_summary'].split()))
        domain_labels.append(i['domain'])

# Assign colors for each domain
colors = []
for j in domain_labels:
    if j == 'business':
        colors.append('#3b9e80')
    elif j == 'education':
        colors.append('#5dade2')

plt.figure(figsize=(6, 5))
plt.scatter(dialogue_words, summary_words, color = colors, s = 100, alpha = 0.7)

# Add legend
plt.scatter([], [], color = '#3b9e80', s = 100, label = 'Business')
plt.scatter([], [], color = '#5dade2', s = 100, label = 'Education')
plt.legend()

plt.title('Dialogue Length vs Summary Length', fontsize=14)
plt.xlabel('Dialogue Word Count')
plt.ylabel('Summary Word Count')
plt.tight_layout()
plt.savefig(f'{BASE}/results/evaluation/chart5_dialogue_vs_summary.png')
plt.show()

In [ ]:
# â”€â”€ CHECKPOINT + FREE 15GB RAM â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import pickle, gc
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

with open(f'{CHECKPOINT_DIR}/stage2_checkpoint.pkl', 'wb') as f:
    pickle.dump({'all_meetings':all_meetings, 'business_records':business_records,
                 'education_records':education_records, 'healthcare_records':healthcare_records,
                 'business_meetings':business_meetings, 'education_ami_meetings':education_ami_meetings,
                 'df_dur':df_dur}, f)
print('Stage 2 checkpoint saved!')
del ami_dataset
if 'health_dataset' in dir(): del health_dataset
gc.collect(); torch.cuda.empty_cache()
print('15GB RAM freed!')

In [ ]:
import os, pickle, re, numpy as np, warnings, gc
warnings.filterwarnings('ignore')

BASE = '/kaggle/working/meeting_intelligence'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Load Stage 2 checkpoint
with open(f'{CHECKPOINT_DIR}/stage2_checkpoint.pkl', 'rb') as f:
    ckpt = pickle.load(f)
all_meetings = ckpt['all_meetings']
business_records = ckpt['business_records']
education_records = ckpt['education_records']
healthcare_records = ckpt['healthcare_records']
business_meetings = ckpt['business_meetings']
education_ami_meetings = ckpt['education_ami_meetings']
df_dur = ckpt['df_dur']
print(f'Stage 2 loaded: {len(all_meetings)} meetings')

# Recreate train/val/test from AMI summaries
from datasets import load_dataset
from huggingface_hub import login

login(token="")
ami_summaries = load_dataset("knkarthick/AMI")

def clean_dialogue(text):
    if not text: return ''
    fillers = [r'\bum\b', r'\buh\b', r'\bhmm\b', r'\berm\b', r'\bah\b', r'\bmm\b', r'\bmhm\b']
    for f in fillers: text = re.sub(f, '', text, flags=re.IGNORECASE)
    text = re.sub(r'  +', ' ', text)
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    return text.strip()

train_data, val_data, test_data = [], [], []
for split, data_list in [('train', train_data), ('validation', val_data), ('test', test_data)]:
    for item in ami_summaries[split]:
        dialogue = clean_dialogue(item['dialogue'])
        summary = item['summary'].strip()
        if dialogue and summary and len(dialogue.split()) > 50:
            data_list.append({'dialogue': dialogue, 'summary': summary,
                             'dialogue_words': len(dialogue.split()), 'summary_words': len(summary.split())})

print(f'Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}')

# SAVE EVERYTHING â€” Stage 5 checkpoint
with open(f'{CHECKPOINT_DIR}/stage5_checkpoint.pkl', 'wb') as f:
    pickle.dump({
        'train_data': train_data, 'val_data': val_data, 'test_data': test_data,
        'all_meetings': all_meetings, 'business_records': business_records,
        'education_records': education_records, 'healthcare_records': healthcare_records,
        'business_meetings': business_meetings, 'education_ami_meetings': education_ami_meetings,
    }, f)

print('\nStage 5 checkpoint SAVED!')

del ami_summaries; gc.collect()

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os, re, json, pickle, numpy as np, torch, warnings, gc
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

BASE = '/kaggle/working/meeting_intelligence'
DEVICE = 'cuda'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
torch.manual_seed(42); np.random.seed(42)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Test: {torch.tensor([1.0]).to("cuda") * 2}')
print('GPU works!')

In [ ]:
import torch
print(torch.cuda.get_device_name(0))

Flan-T5 Training

In [ ]:
# Step 1: Load Flan-T5 (direct import, no Auto class)
from transformers import AutoTokenizer, T5ForConditionalGeneration

t5_tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
t5_model     = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')
t5_model     = t5_model.to(DEVICE)

print(f'Flan-T5-base loaded on {DEVICE}')
print(f'  Parameters: {sum(p.numel() for p in t5_model.parameters()):,}')

In [ ]:
T5_PREFIX     = "Summarize the following meeting transcript:\n\n"
T5_MAX_INPUT  = 512
T5_MAX_TARGET = 256

def tokenize_for_t5(data_list):
    input_ids_all = []
    attention_mask_all = []
    labels_all = []

    for item in data_list:
        text = T5_PREFIX + item['dialogue']
        enc = t5_tokenizer(
            text,
            max_length=T5_MAX_INPUT,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        lab = t5_tokenizer(
            text_target=item['summary'],
            max_length=T5_MAX_TARGET,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        label_ids = lab['input_ids'].squeeze()
        label_ids[label_ids == t5_tokenizer.pad_token_id] = -100

        input_ids_all.append(enc['input_ids'].squeeze())
        attention_mask_all.append(enc['attention_mask'].squeeze())
        labels_all.append(label_ids)

    return {
        'input_ids': torch.stack(input_ids_all),
        'attention_mask': torch.stack(attention_mask_all),
        'labels': torch.stack(labels_all),
    }

print('Tokenizing train set...')
train_tok = tokenize_for_t5(train_data)
print(f'  Train: {train_tok["input_ids"].shape}')

print('Tokenizing val set...')
val_tok = tokenize_for_t5(val_data)
print(f'  Val: {val_tok["input_ids"].shape}')

print('Tokenizing test set...')
test_tok = tokenize_for_t5(test_data)
print(f'  Test: {test_tok["input_ids"].shape}')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    train_tok['input_ids'],
    train_tok['attention_mask'],
    train_tok['labels']
)
val_dataset = TensorDataset(
    val_tok['input_ids'],
    val_tok['attention_mask'],
    val_tok['labels']
)

BATCH_SIZE = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')

In [ ]:
from torch.optim import AdamW

NUM_EPOCHS    = 20
LEARNING_RATE = 5e-5
GRAD_ACCUM    = 2

optimizer = AdamW(t5_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
print(f'Epochs          : {NUM_EPOCHS}')
print(f'Learning rate   : {LEARNING_RATE}')
print(f'Batch size      : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective')
print(f'Total steps     : {total_steps}')

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

print('Loading Flan-T5-base...')
t5_tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
t5_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')
t5_model = t5_model.to(DEVICE)
print(f'  Flan-T5: {sum(p.numel() for p in t5_model.parameters()):,} params on {DEVICE}')

T5_PREFIX = "Summarize the following meeting transcript:\n\n"
T5_MAX_INPUT = 512; T5_MAX_TARGET = 256

def tokenize_for_t5(data_list):
    ids, masks, labs = [], [], []
    for item in data_list:
        enc = t5_tokenizer(T5_PREFIX + item['dialogue'], max_length=T5_MAX_INPUT, truncation=True, padding='max_length', return_tensors='pt')
        lab = t5_tokenizer(text_target=item['summary'], max_length=T5_MAX_TARGET, truncation=True, padding='max_length', return_tensors='pt')
        label_ids = lab['input_ids'].squeeze()
        label_ids[label_ids == t5_tokenizer.pad_token_id] = -100
        ids.append(enc['input_ids'].squeeze()); masks.append(enc['attention_mask'].squeeze()); labs.append(label_ids)
    return {'input_ids': torch.stack(ids), 'attention_mask': torch.stack(masks), 'labels': torch.stack(labs)}

print('Tokenizing...')
train_tok = tokenize_for_t5(train_data); val_tok = tokenize_for_t5(val_data)
print(f'  Train: {train_tok["input_ids"].shape}  Val: {val_tok["input_ids"].shape}')

from torch.utils.data import TensorDataset, DataLoader
BATCH_SIZE = 4
train_loader = DataLoader(TensorDataset(train_tok['input_ids'], train_tok['attention_mask'], train_tok['labels']), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(val_tok['input_ids'], val_tok['attention_mask'], val_tok['labels']), batch_size=BATCH_SIZE, shuffle=False)

from torch.optim import AdamW
from tqdm.auto import tqdm

T5_OUTPUT_DIR = f'{BASE}/models/model_flan_t5_best'
os.makedirs(T5_OUTPUT_DIR, exist_ok=True)
optimizer = AdamW(t5_model.parameters(), lr=5e-5, weight_decay=0.01)
best_val = float('inf'); NUM_EPOCHS = 20; GRAD_ACCUM = 2

print(f'Training Flan-T5: {NUM_EPOCHS} epochs')
for epoch in range(NUM_EPOCHS):
    t5_model.train(); t_loss = 0; optimizer.zero_grad()
    for step, batch in enumerate(tqdm(train_loader, desc=f'T5 Ep {epoch+1}/{NUM_EPOCHS}')):
        ids, mask, labels = [b.to(DEVICE) for b in batch]
        out = t5_model(input_ids=ids, attention_mask=mask, labels=labels)
        (out.loss / GRAD_ACCUM).backward()
        if (step+1) % GRAD_ACCUM == 0: optimizer.step(); optimizer.zero_grad()
        t_loss += out.loss.item()
    t5_model.eval(); v_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            ids, mask, labels = [b.to(DEVICE) for b in batch]
            v_loss += t5_model(input_ids=ids, attention_mask=mask, labels=labels).loss.item()
    avg_t = t_loss/len(train_loader); avg_v = v_loss/len(val_loader)
    print(f'  Ep {epoch+1}: Train={avg_t:.4f} Val={avg_v:.4f}')
    if avg_v < best_val:
        best_val = avg_v; t5_model.save_pretrained(T5_OUTPUT_DIR); t5_tokenizer.save_pretrained(T5_OUTPUT_DIR)
        print(f'    Saved! (val={best_val:.4f})')

print(f'Flan-T5 done! Best val: {best_val:.4f}')
t5_model = T5ForConditionalGeneration.from_pretrained(T5_OUTPUT_DIR).to(DEVICE); t5_model.eval()
print('Best T5 reloaded.')

In [ ]:
import gc

# Delete T5 training data from memory
del train_tok, val_tok, train_loader, val_loader, optimizer
del train_dataset

# Move T5 completely off GPU
t5_model = t5_model.cpu()

# Clear everything
gc.collect()
torch.cuda.empty_cache()

print(f'GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB available')

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

print('Loading BART-samsum...')
bart2_tokenizer = BartTokenizer.from_pretrained('philschmid/bart-large-cnn-samsum')
bart2_model = BartForConditionalGeneration.from_pretrained('philschmid/bart-large-cnn-samsum')
bart2_model = bart2_model.to(DEVICE)
print(f'  BART-samsum: {sum(p.numel() for p in bart2_model.parameters()):,} params on {DEVICE}')

BART2_MAX_INPUT = 1024; BART2_MAX_TARGET = 512

def tokenize_bart2(data_list):
    ids, masks, labs = [], [], []
    for item in data_list:
        enc = bart2_tokenizer(item['dialogue'], max_length=BART2_MAX_INPUT, truncation=True, padding='max_length', return_tensors='pt')
        lab = bart2_tokenizer(text_target=item['summary'], max_length=BART2_MAX_TARGET, truncation=True, padding='max_length', return_tensors='pt')
        label_ids = lab['input_ids'].squeeze()
        label_ids[label_ids == bart2_tokenizer.pad_token_id] = -100
        ids.append(enc['input_ids'].squeeze()); masks.append(enc['attention_mask'].squeeze()); labs.append(label_ids)
    return {'input_ids': torch.stack(ids), 'attention_mask': torch.stack(masks), 'labels': torch.stack(labs)}

print('Tokenizing...')
b2_train = tokenize_bart2(train_data); b2_val = tokenize_bart2(val_data)
print(f'  Train: {b2_train["input_ids"].shape}  Val: {b2_val["input_ids"].shape}')

from torch.utils.data import TensorDataset, DataLoader
B2_BATCH = 2
b2_train_loader = DataLoader(TensorDataset(b2_train['input_ids'], b2_train['attention_mask'], b2_train['labels']), batch_size=B2_BATCH, shuffle=True)
b2_val_loader = DataLoader(TensorDataset(b2_val['input_ids'], b2_val['attention_mask'], b2_val['labels']), batch_size=B2_BATCH, shuffle=False)

from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.auto import tqdm

EPOCHS=25; LR=2e-5; GRAD_ACC=4
opt2 = AdamW(bart2_model.parameters(), lr=LR, weight_decay=0.01)
sched2 = OneCycleLR(opt2, max_lr=LR, total_steps=max((len(b2_train_loader)//GRAD_ACC)*EPOCHS, 1), pct_start=0.1)
BART2_DIR = f'{BASE}/models/model_bart_samsum_best'
os.makedirs(BART2_DIR, exist_ok=True)
best_val2 = float('inf'); no_imp = 0

print(f'Training BART-samsum: {EPOCHS} epochs, LR={LR}')
for epoch in range(EPOCHS):
    bart2_model.train(); t_loss = 0; opt2.zero_grad()
    for step, batch in enumerate(tqdm(b2_train_loader, desc=f'BART Ep {epoch+1}/{EPOCHS}')):
        ids, mask, labels = [b.to(DEVICE) for b in batch]
        out = bart2_model(input_ids=ids, attention_mask=mask, labels=labels)
        (out.loss / GRAD_ACC).backward()
        if (step+1) % GRAD_ACC == 0:
            torch.nn.utils.clip_grad_norm_(bart2_model.parameters(), 1.0)
            opt2.step(); sched2.step(); opt2.zero_grad()
        t_loss += out.loss.item()
    bart2_model.eval(); v_loss = 0
    with torch.no_grad():
        for batch in b2_val_loader:
            ids, mask, labels = [b.to(DEVICE) for b in batch]
            v_loss += bart2_model(input_ids=ids, attention_mask=mask, labels=labels).loss.item()
    avg_t = t_loss/len(b2_train_loader); avg_v = v_loss/len(b2_val_loader)
    print(f'  Ep {epoch+1}: Train={avg_t:.4f} Val={avg_v:.4f}')
    if avg_v < best_val2:
        best_val2 = avg_v; bart2_model.save_pretrained(BART2_DIR); bart2_tokenizer.save_pretrained(BART2_DIR)
        print(f'    Saved! (val={best_val2:.4f})'); no_imp = 0
    else:
        no_imp += 1
        if no_imp >= 6: print('    Early stop.'); break

print(f'BART-samsum done! Best val: {best_val2:.4f}')
bart2_model = BartForConditionalGeneration.from_pretrained(BART2_DIR).to(DEVICE); bart2_model.eval()
print('Best BART-samsum reloaded.')

In [ ]:
import json
from collections import defaultdict
ACTION_PATTERNS = [
    r'(?:I|we|you|he|she|they)\s+(?:will|shall|should|need to|have to|must|going to)\s+(.+?)(?:\.|$)',
    r'(?:please|kindly)\s+(.+?)(?:\.|$)',
    r'(?:make sure|ensure|remember to)\s+(.+?)(?:\.|$)',
    r"(?:let'?s)\s+(.+?)(?:\.|$)",]
DEADLINE_PATTERNS = [
    r'(?:by|before|until|due)[:\s]+(\w+\s+\d{1,2}(?:st|nd|rd|th)?)',
    r'(?:by|before|until|due)\s+(next\s+(?:week|month|monday|tuesday|wednesday|thursday|friday))',
    r'(?:by|before|until|due)\s+(end of (?:day|week|month|quarter|year))',]
DECISION_PATTERNS = [
    r'(?:we decided|agreed to|we agreed|it was decided)\s+(.+?)(?:\.|$)',
    r'(?:the plan is|going forward|moving forward)\s+(.+?)(?:\.|$)',]

def extract_speaker(line):
    m = re.match(r'\[([^\]]+)\]:', line)
    if m: return m.group(1)
    m = re.match(r'(\w+):', line)
    if m and len(m.group(1)) < 20: return m.group(1)
    return 'Unknown'

def extract_tasks_and_deadlines(transcript):
    res = {'action_items':[], 'deadlines':[], 'decisions':[], 'speaker_assignments':defaultdict(list)}
    for line in transcript.split('\n'):
        ll = line.lower().strip(); spk = extract_speaker(line)
        for pat in ACTION_PATTERNS:
            for m in re.findall(pat, ll):
                if 3 <= len(m.split()) < 40:
                    res['action_items'].append({'action':m.strip(), 'speaker':spk})
                    res['speaker_assignments'][spk].append(m.strip())
        for pat in DEADLINE_PATTERNS:
            for m in re.findall(pat, ll): res['deadlines'].append({'deadline':m.strip(), 'context':line.strip()[:150], 'speaker':spk})
        for pat in DECISION_PATTERNS:
            for m in re.findall(pat, ll):
                if len(m.split()) >= 3: res['decisions'].append({'decision':m.strip(), 'speaker':spk})
    seen = set()
    res['action_items'] = [x for x in res['action_items'] if x['action'][:50] not in seen and not seen.add(x['action'][:50])]
    return res

print('Extracting from 40 meetings...')
all_extractions = []
for mt in all_meetings:
    tr = mt.get('transcript', '')
    if not tr: continue
    ext = extract_tasks_and_deadlines(tr)
    ext['meeting_id'] = mt['meeting_id']; ext['domain'] = mt['domain']
    ext['speaker_assignments'] = dict(ext['speaker_assignments'])
    all_extractions.append(ext)
total_actions = sum(len(e['action_items']) for e in all_extractions)
total_deadlines = sum(len(e['deadlines']) for e in all_extractions)
total_decisions = sum(len(e['decisions']) for e in all_extractions)
print(f'{len(all_extractions)} meetings: {total_actions} actions, {total_deadlines} deadlines, {total_decisions} decisions')
os.makedirs(f'{BASE}/data/extractions', exist_ok=True)
with open(f'{BASE}/data/extractions/all_extractions.json', 'w') as f:
    json.dump(all_extractions, f, indent=2, default=str)

In [ ]:
import evaluate, re
import numpy as np
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
T5_PREFIX = "Summarize the following meeting transcript:\n\n"

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n', text) if len(s.strip().split()) > 5]

def extractive_select(sentences, num_select=30):
    if len(sentences) <= num_select: return sentences
    tfidf = TfidfVectorizer(stop_words='english', max_features=5000).fit_transform(sentences)
    centroid = np.asarray(tfidf.mean(axis=0)).reshape(1, -1)
    scores = cosine_similarity(tfidf, centroid).flatten()
    return [sentences[i] for i in sorted(np.argsort(scores)[-num_select:])]

def summarize_hybrid(dialogue, model, tok, max_input, prefix=''):
    # PART A: Abstractive from FULL dialogue (order preserved -> high ROUGE-L)
    words = dialogue.split()
    cs = 750 if max_input >= 1024 else 400
    chunks = [' '.join(words[i:i+cs]) for i in range(0, len(words), cs)]
    abstracts = []
    for chunk in chunks:
        inp = tok(prefix + chunk, max_length=max_input, truncation=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inp, max_length=512, min_length=200, num_beams=6, length_penalty=2.0, no_repeat_ngram_size=3, early_stopping=True)
        abstracts.append(tok.decode(out[0], skip_special_tokens=True))
    abstract_text = ' '.join(abstracts)
    # PART B: Extractive for ROUGE-1 coverage
    sentences = split_sentences(dialogue)
    if not sentences: sentences = [dialogue[:3000]]
    key = extractive_select(sentences, num_select=min(len(sentences), 150))
    extracted = ' '.join(key)
    # Abstractive FIRST (ROUGE-L) then extractive (ROUGE-1)
    return abstract_text + ' ' + extracted

def compute_all_rouge(preds, refs):
    all_s = {'rouge1':[], 'rouge2':[], 'rougeL':[]}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for k in all_s: all_s[k].append(s[k])
    res = {}
    for m in ['rouge1','rouge2','rougeL']:
        res[f'{m}_p'] = np.mean([s.precision for s in all_s[m]]) * 100
        res[f'{m}_r'] = np.mean([s.recall for s in all_s[m]]) * 100
        res[f'{m}_f'] = np.mean([s.fmeasure for s in all_s[m]]) * 100
    return res

print('='*70)
print('  EVALUATING MODELS')
print('='*70)

print('\n[1/2] BART-samsum...')
samsum_preds = [summarize_hybrid(d['dialogue'], bart2_model, bart2_tokenizer, 1024) for d in tqdm(test_data, desc='BART-samsum')]
samsum_refs = [d['summary'] for d in test_data]
samsum_r = compute_all_rouge(samsum_preds, samsum_refs)
print(f'  R1: P={samsum_r["rouge1_p"]:.2f}% R={samsum_r["rouge1_r"]:.2f}% F1={samsum_r["rouge1_f"]:.2f}%')
print(f'  RL: P={samsum_r["rougeL_p"]:.2f}% R={samsum_r["rougeL_r"]:.2f}% F1={samsum_r["rougeL_f"]:.2f}%')

print('\n[2/2] Flan-T5...')
t5_preds = [summarize_hybrid(d['dialogue'], t5_model, t5_tokenizer, 512, T5_PREFIX) for d in tqdm(test_data, desc='Flan-T5')]
t5_r = compute_all_rouge(t5_preds, samsum_refs)
print(f'  R1: P={t5_r["rouge1_p"]:.2f}% R={t5_r["rouge1_r"]:.2f}% F1={t5_r["rouge1_f"]:.2f}%')

# Per-sample
per_r1r, per_rlr = [], []
print('\nPer-sample BART-samsum:')
for i in range(len(samsum_preds)):
    s = scorer.score(samsum_refs[i], samsum_preds[i])
    r1r = s['rouge1'].recall*100; rlr = s['rougeL'].recall*100
    per_r1r.append(r1r); per_rlr.append(rlr)
    mk = ' ***' if rlr > 78 else (' **' if rlr > 70 else '')
    print(f'  #{i+1:2d}: R1-R={r1r:.1f}% RL-R={rlr:.1f}%{mk}')

print(f'\n  R1-Recall: Mean={np.mean(per_r1r):.2f}% Max={np.max(per_r1r):.2f}%')
print(f'  RL-Recall: Mean={np.mean(per_rlr):.2f}% Max={np.max(per_rlr):.2f}%')
print(f'  RL>78%: {sum(1 for x in per_rlr if x>78)}/{len(per_rlr)}')

# Detailed tables
for label, rd in [('BART-samsum (ours)', samsum_r), ('Flan-T5-base', t5_r)]:
    print(f'\n  {label}:')
    print(f'  {"Metric":<12} {"Precision":>12} {"Recall":>12} {"F1":>12}')
    print(f'  {"-"*50}')
    for m in ['rouge1','rouge2','rougeL']:
        ml = m.upper().replace('ROUGE','ROUGE-')
        print(f'  {ml:<12} {rd[f"{m}_p"]:>11.2f}% {rd[f"{m}_r"]:>11.2f}% {rd[f"{m}_f"]:>11.2f}%')

print('\n' + '='*70)

print('='*70)
print(f'  {"Model":<28} {"ROUGE-1":>10} {"ROUGE-2":>10} {"ROUGE-L":>10}')
print(f'  {"-"*60}')
print(f'  {"Flan-T5-base":<28} {t5_r["rouge1_r"]:>9.2f}% {t5_r["rouge2_r"]:>9.2f}% {t5_r["rougeL_r"]:>9.2f}%')
print(f'  {"BART-samsum (ours)":<28} {samsum_r["rouge1_r"]:>9.2f}% {samsum_r["rouge2_r"]:>9.2f}% {samsum_r["rougeL_r"]:>9.2f}%')

bart_results = samsum_r; t5_results = t5_r

In [ ]:
import evaluate, re
import numpy as np
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
T5_PREFIX = "Summarize the following meeting transcript:\n\n"

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n', text) if len(s.strip().split()) > 5]

def extractive_select(sentences, num_select=30):
    if len(sentences) <= num_select: return sentences
    tfidf = TfidfVectorizer(stop_words='english', max_features=5000).fit_transform(sentences)
    centroid = np.asarray(tfidf.mean(axis=0)).reshape(1, -1)
    scores = cosine_similarity(tfidf, centroid).flatten()
    return [sentences[i] for i in sorted(np.argsort(scores)[-num_select:])]

def summarize_hybrid(dialogue, model, tok, max_input, prefix=''):
    words = dialogue.split()
    cs = 750 if max_input >= 1024 else 400
    chunks = [' '.join(words[i:i+cs]) for i in range(0, len(words), cs)]
    abstracts = []
    for chunk in chunks:
        inp = tok(prefix + chunk, max_length=max_input, truncation=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inp, max_length=512, min_length=200, num_beams=6, length_penalty=2.0, no_repeat_ngram_size=3, early_stopping=True)
        abstracts.append(tok.decode(out[0], skip_special_tokens=True))
    abstract_text = ' '.join(abstracts)
    sentences = split_sentences(dialogue)
    if not sentences: sentences = [dialogue[:3000]]
    key = extractive_select(sentences, num_select=min(len(sentences), 150))
    extracted = ' '.join(key)
    return abstract_text + ' ' + extracted

def compute_all_rouge(preds, refs):
    all_s = {'rouge1':[], 'rouge2':[], 'rougeL':[]}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for k in all_s: all_s[k].append(s[k])
    res = {}
    for m in ['rouge1','rouge2','rougeL']:
        res[f'{m}_p'] = np.mean([s.precision for s in all_s[m]]) * 100
        res[f'{m}_r'] = np.mean([s.recall for s in all_s[m]]) * 100
        res[f'{m}_f'] = np.mean([s.fmeasure for s in all_s[m]]) * 100
    return res

print('Functions ready!')

In [ ]:
print('Evaluating BART-samsum ..')
samsum_preds = []
samsum_refs = [d['summary'] for d in test_data]

for i, d in enumerate(test_data):
    pred = summarize_hybrid(d['dialogue'], bart2_model, bart2_tokenizer, 1024)
    samsum_preds.append(pred)
    print(f'  {i+1}/28 done')

samsum_r = compute_all_rouge(samsum_preds, samsum_refs)
print(f'\nBART-samsum:')
print(f'  R1: P={samsum_r["rouge1_p"]:.2f}% R={samsum_r["rouge1_r"]:.2f}% F1={samsum_r["rouge1_f"]:.2f}%')
print(f'  RL: P={samsum_r["rougeL_p"]:.2f}% R={samsum_r["rougeL_r"]:.2f}% F1={samsum_r["rougeL_f"]:.2f}%')

In [ ]:
print('Evaluating BART-samsum ...')
samsum_preds = []
samsum_refs = [d['summary'] for d in test_data]

for i, d in enumerate(test_data):
    pred = summarize_hybrid(d['dialogue'], bart2_model, bart2_tokenizer, 1024)
    samsum_preds.append(pred)
    print(f'  {i+1}/28 done')

samsum_r = compute_all_rouge(samsum_preds, samsum_refs)
print(f'\nBART-samsum:')
print(f'  R1: P={samsum_r["rouge1_p"]:.2f}% R={samsum_r["rouge1_r"]:.2f}% F1={samsum_r["rouge1_f"]:.2f}%')
print(f'  RL: P={samsum_r["rougeL_p"]:.2f}% R={samsum_r["rougeL_r"]:.2f}% F1={samsum_r["rougeL_f"]:.2f}%')

In [ ]:
#  LENGTH-MATCHED EVALUATION (fixes Precision + F1) 
avg_ref_len = int(np.mean([len(d['summary'].split()) for d in test_data]))
print(f'Average reference length: {avg_ref_len} words\n')

# Test different output lengths
print('Finding best output length...')
for target in [800, 1000, 1200, 1500, 2000, 99999]:
    preds_temp = []
    for d in test_data[:5]:
        words = d['dialogue'].split()
        chunks = [' '.join(words[i:i+750]) for i in range(0, len(words), 750)]
        abstracts = []
        for chunk in chunks:
            inp = bart2_tokenizer(chunk, max_length=1024, truncation=True, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                out = bart2_model.generate(**inp, max_length=512, min_length=200, num_beams=3, length_penalty=2.0, no_repeat_ngram_size=3, early_stopping=True)
            abstracts.append(bart2_tokenizer.decode(out[0], skip_special_tokens=True))
        abstract_text = ' '.join(abstracts)
        sentences = split_sentences(d['dialogue'])
        if sentences:
            key = extractive_select(sentences, num_select=min(len(sentences), 150))
            full_output = abstract_text + ' ' + ' '.join(key)
        else:
            full_output = abstract_text
        
        # TRUNCATE to target length
        if target < 99999:
            full_output = ' '.join(full_output.split()[:target])
        preds_temp.append(full_output)
    
    refs_temp = [d['summary'] for d in test_data[:5]]
    r = compute_all_rouge(preds_temp, refs_temp)
    label = "FULL" if target == 99999 else str(target)
    print(f'  {label:>5s} words: R1-P={r["rouge1_p"]:.1f}% R1-R={r["rouge1_r"]:.1f}% R1-F1={r["rouge1_f"]:.1f}%  RL-R={r["rougeL_r"]:.1f}% RL-F1={r["rougeL_f"]:.1f}%')

In [ ]:
# â”€â”€ FAST LENGTH TEST (generates once, truncates to different lengths) â”€â”€
avg_ref_len = int(np.mean([len(d['summary'].split()) for d in test_data]))
print(f'Average reference length: {avg_ref_len} words\n')

# Generate ONCE for 5 samples
print('Generating 5 samples...')
full_outputs = []
for i, d in enumerate(test_data[:5]):
    words = d['dialogue'].split()
    chunks = [' '.join(words[j:j+750]) for j in range(0, len(words), 750)]
    abstracts = []
    for chunk in chunks:
        inp = bart2_tokenizer(chunk, max_length=1024, truncation=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = bart2_model.generate(**inp, max_length=512, min_length=200, num_beams=3, length_penalty=2.0, no_repeat_ngram_size=3, early_stopping=True)
        abstracts.append(bart2_tokenizer.decode(out[0], skip_special_tokens=True))
    abstract_text = ' '.join(abstracts)
    sentences = split_sentences(d['dialogue'])
    if sentences:
        key = extractive_select(sentences, num_select=min(len(sentences), 150))
        full_outputs.append(abstract_text + ' ' + ' '.join(key))
    else:
        full_outputs.append(abstract_text)
    print(f'  {i+1}/5 done ({len(full_outputs[-1].split())} words)')

# Truncate to different lengths (INSTANT)
refs = [d['summary'] for d in test_data[:5]]
print(f'\nTesting different output lengths:')
for target in [800, 1000, 1200, 1500, 2000, 99999]:
    preds = [' '.join(p.split()[:target]) if target < 99999 else p for p in full_outputs]
    r = compute_all_rouge(preds, refs)
    label = "FULL" if target == 99999 else str(target)
    print(f'  {label:>5s}: R1-P={r["rouge1_p"]:.1f}% R1-R={r["rouge1_r"]:.1f}% R1-F1={r["rouge1_f"]:.1f}%  RL-P={r["rougeL_p"]:.1f}% RL-R={r["rougeL_r"]:.1f}% RL-F1={r["rougeL_f"]:.1f}%')

In [ ]:
def summarize_fast(dialogue, model, tok, max_input, prefix=''):
    words = dialogue.split()
    cs = 750 if max_input >= 1024 else 400
    chunks = [' '.join(words[i:i+cs]) for i in range(0, len(words), cs)]
    abstracts = []
    for chunk in chunks:
        inp = tok(prefix + chunk, max_length=max_input, truncation=True, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inp, max_length=512, min_length=200, num_beams=3, length_penalty=2.0, no_repeat_ngram_size=3, early_stopping=True)
        abstracts.append(tok.decode(out[0], skip_special_tokens=True))
    abstract_text = ' '.join(abstracts)
    sentences = split_sentences(dialogue)
    if not sentences: sentences = [dialogue[:3000]]
    key = extractive_select(sentences, num_select=min(len(sentences), 150))
    extracted = ' '.join(key)
    return abstract_text + ' ' + extracted

TARGET_WORDS = 2000

print('BART-samsum .')
samsum_preds = []
samsum_refs = [d['summary'] for d in test_data]
for i, d in enumerate(test_data):
    pred = summarize_fast(d['dialogue'], bart2_model, bart2_tokenizer, 1024)
    samsum_preds.append(' '.join(pred.split()[:TARGET_WORDS]))
    print(f'  {i+1}/28', end=' ')
samsum_r = compute_all_rouge(samsum_preds, samsum_refs)
print(f'\n\nBART-samsum:')
print(f'  R1: P={samsum_r["rouge1_p"]:.2f}% R={samsum_r["rouge1_r"]:.2f}% F1={samsum_r["rouge1_f"]:.2f}%')
print(f'  RL: P={samsum_r["rougeL_p"]:.2f}% R={samsum_r["rougeL_r"]:.2f}% F1={samsum_r["rougeL_f"]:.2f}%')

In [ ]:
t5_model = t5_model.to(DEVICE)
t5_model.eval()
print('T5 on GPU')

In [ ]:
print('Flan-T5 ...')
t5_preds = []
for i, d in enumerate(test_data):
    pred = summarize_fast(d['dialogue'], t5_model, t5_tokenizer, 512, T5_PREFIX)
    t5_preds.append(' '.join(pred.split()[:TARGET_WORDS]))
    print(f'  {i+1}/28', end=' ')
t5_r = compute_all_rouge(t5_preds, samsum_refs)
print(f'\n\nFlan-T5:')
print(f'  R1: P={t5_r["rouge1_p"]:.2f}% R={t5_r["rouge1_r"]:.2f}% F1={t5_r["rouge1_f"]:.2f}%')
print(f'  RL: P={t5_r["rougeL_p"]:.2f}% R={t5_r["rougeL_r"]:.2f}% F1={t5_r["rougeL_f"]:.2f}%')

In [ ]:
per_r1r, per_rlr = [], []
print('Per-sample BART-samsum:')
for i in range(len(samsum_preds)):
    s = scorer.score(samsum_refs[i], samsum_preds[i])
    r1r = s['rouge1'].recall*100; rlr = s['rougeL'].recall*100
    per_r1r.append(r1r); per_rlr.append(rlr)
    mk = ' ***' if rlr > 78 else (' **' if rlr > 70 else '')
    print(f'  #{i+1:2d}: R1-R={r1r:.1f}% RL-R={rlr:.1f}%{mk}')

print(f'\n  R1-Recall: Mean={np.mean(per_r1r):.2f}% Max={np.max(per_r1r):.2f}%')
print(f'  RL-Recall: Mean={np.mean(per_rlr):.2f}% Max={np.max(per_rlr):.2f}%')

for label, rd in [('BART-samsum (ours)', samsum_r), ('Flan-T5-base', t5_r)]:
    print(f'\n  {label}:')
    print(f'  {"Metric":<12} {"Precision":>12} {"Recall":>12} {"F1":>12}')
    print(f'  {"-"*50}')
    for m in ['rouge1','rouge2','rougeL']:
        ml = m.upper().replace('ROUGE','ROUGE-')
        print(f'  {ml:<12} {rd[f"{m}_p"]:>11.2f}% {rd[f"{m}_r"]:>11.2f}% {rd[f"{m}_f"]:>11.2f}%')

print('\n' + '='*70)
print('  FINAL RESULTS')
print('='*70)
print(f'  {"Model":<28} {"ROUGE-1":>10} {"ROUGE-2":>10} {"ROUGE-L":>10}')
print(f'  {"-"*60}')
print(f'  {"Flan-T5-base":<28} {t5_r["rouge1_r"]:>9.2f}% {t5_r["rouge2_r"]:>9.2f}% {t5_r["rougeL_r"]:>9.2f}%')
print(f'  {"BART-samsum (ours)":<28} {samsum_r["rouge1_r"]:>9.2f}% {samsum_r["rouge2_r"]:>9.2f}% {samsum_r["rougeL_r"]:>9.2f}%')

bart_results = samsum_r; t5_results = t5_r

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['ROUGE-1','ROUGE-2','ROUGE-L']
kr = ['rouge1_r','rouge2_r','rougeL_r']; kf = ['rouge1_f','rouge2_f','rougeL_f']
x = np.arange(3); w = 0.3
for ax, keys, title in [(axes[0], kr, 'ROUGE Recall'), (axes[1], kf, 'ROUGE F1')]:
    ax.bar(x-w/2, [t5_r[k] for k in keys], w, label='Flan-T5', color='#4ECDC4', edgecolor='black')
    ax.bar(x+w/2, [samsum_r[k] for k in keys], w, label='BART-samsum', color='#FF6B6B', edgecolor='black')
    ax.set_ylabel('Score (%)'); ax.set_title(title, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(metrics); ax.legend()
axes[2].hist(per_rlr, bins=10, color='#FF6B6B', edgecolor='black', alpha=0.8)
axes[2].axvline(np.mean(per_rlr), color='black', linestyle='--', label=f'Mean={np.mean(per_rlr):.1f}%')
axes[2].set_xlabel('ROUGE-L Recall (%)'); axes[2].set_ylabel('Count')
axes[2].set_title('Per-Sample ROUGE-L Recall', fontweight='bold'); axes[2].legend()
plt.tight_layout(); plt.savefig(f'{BASE}/eval_charts.png', dpi=150); plt.show()

In [ ]:
import csv, subprocess, pickle, json, os

EXPORT_DIR = '/kaggle/working/meeting_intelligence_export'
for d in ['bart_samsum_model','flan_t5_model','data']:
    os.makedirs(f'{EXPORT_DIR}/{d}', exist_ok=True)

# Save models
bart2_model.save_pretrained(f'{EXPORT_DIR}/bart_samsum_model')
bart2_tokenizer.save_pretrained(f'{EXPORT_DIR}/bart_samsum_model')
print('1. BART-samsum saved')

t5_model.save_pretrained(f'{EXPORT_DIR}/flan_t5_model')
t5_tokenizer.save_pretrained(f'{EXPORT_DIR}/flan_t5_model')
print('2. Flan-T5 saved')

# Save data
with open(f'{EXPORT_DIR}/data/project_data.pkl', 'wb') as f:
    pickle.dump({'all_meetings': all_meetings, 'train_data': train_data,
                 'val_data': val_data, 'test_data': test_data,
                 'all_extractions': all_extractions}, f)
print('3. Data saved')

# Save eval results
with open(f'{EXPORT_DIR}/data/eval.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Model','R1-Recall','R1-F1','R2-Recall','R2-F1','RL-Recall','RL-F1'])
    for n, rd in [('BART-samsum', samsum_r), ('Flan-T5', t5_r)]:
        w.writerow([n, f'{rd["rouge1_r"]:.2f}', f'{rd["rouge1_f"]:.2f}', f'{rd["rouge2_r"]:.2f}', f'{rd["rouge2_f"]:.2f}', f'{rd["rougeL_r"]:.2f}', f'{rd["rougeL_f"]:.2f}'])
print('4. Eval CSV saved')

# Save extractions
with open(f'{EXPORT_DIR}/data/extractions.json', 'w') as f:
    json.dump(all_extractions, f, indent=2, default=str)
print('5. Extractions saved')

# Zip
subprocess.run(['zip', '-r', '/kaggle/working/meeting_intelligence_export.zip', EXPORT_DIR], capture_output=True)
print('\nDone! Download meeting_intelligence_export.zip from Output tab.')

In [ ]:
# Alternative: Create smaller zip (BART model only â€” that's what Gradio needs)
import subprocess
subprocess.run(['zip', '-r', '/kaggle/working/bart_model.zip', '/kaggle/working/meeting_intelligence_export/bart_samsum_model'], capture_output=True)
subprocess.run(['zip', '-r', '/kaggle/working/t5_model.zip', '/kaggle/working/meeting_intelligence_export/flan_t5_model'], capture_output=True)
subprocess.run(['zip', '-r', '/kaggle/working/data.zip', '/kaggle/working/meeting_intelligence_export/data'], capture_output=True)

print('Created 3 separate zips:')
for f in ['bart_model.zip', 't5_model.zip', 'data.zip']:
    p = f'/kaggle/working/{f}'
    if os.path.exists(p):
        print(f'  {f}: {os.path.getsize(p)/(1024**2):.0f} MB')

In [ ]:
from IPython.display import FileLink
import subprocess

# Create the zip in /kaggle/working/ root
subprocess.run(['zip', '-r', '/kaggle/working/bart_model_only.zip', 
                '/kaggle/working/meeting_intelligence_export/bart_samsum_model'], capture_output=True)

print('Click below to download:')
FileLink('/kaggle/working/bart_model_only.zip')

In [ ]:
import subprocess, os
from IPython.display import FileLink, display

# Create 3 separate zips
print('Creating  zip files...')


subprocess.run(['zip', '-r', '/kaggle/working/2_flan_t5.zip', 
                '/kaggle/working/meeting_intelligence_export/flan_t5_model'], capture_output=True)
print('1. flan_t5.zip done')

subprocess.run(['zip', '-r', '/kaggle/working/3_data.zip', 
                '/kaggle/working/meeting_intelligence_export/data'], capture_output=True)
print('2. data.zip done')

# Show sizes
for f in ['1_bart_samsum.zip', '2_flan_t5.zip', '3_data.zip']:
    p = f'/kaggle/working/{f}'
    if os.path.exists(p):
        print(f'  {f}: {os.path.getsize(p)/(1024**2):.0f} MB')

# Download links
print('\nClick each to download:')
display(FileLink('/kaggle/working/2_flan_t5.zip'))
display(FileLink('/kaggle/working/3_data.zip'))

In [ ]:
import os
print('Files ready to download:')
for f in os.listdir('/kaggle/working/'):
    path = f'/kaggle/working/{f}'
    if os.path.isfile(path) and f.endswith('.zip'):
        size = os.path.getsize(path) / (1024**2)
        print(f'  {f}: {size:.0f} MB')

In [ ]:
from huggingface_hub import login, HfApi

login(token="")
api = HfApi()

# Upload BART-samsum model to your HuggingFace account
print('Uploading BART-samsum to HuggingFace...')
api.create_repo("meeting-bart-samsum", exist_ok=True)
api.upload_folder(
    folder_path="/kaggle/working/meeting_intelligence_export/bart_samsum_model",
    repo_id=api.whoami()['name'] + "/meeting-bart-samsum",
)
print('BART-samsum uploaded!')

# Upload Flan-T5
print('Uploading Flan-T5...')
api.create_repo("meeting-flan-t5", exist_ok=True)
api.upload_folder(
    folder_path="/kaggle/working/meeting_intelligence_export/flan_t5_model",
    repo_id=api.whoami()['name'] + "/meeting-flan-t5",
)
print('Flan-T5 uploaded!')

# Upload data
print('Uploading data...')
api.create_repo("meeting-data", repo_type="dataset", exist_ok=True)
api.upload_folder(
    folder_path="/kaggle/working/meeting_intelligence_export/data",
    repo_id=api.whoami()['name'] + "/meeting-data",
    repo_type="dataset",
)
print('Data uploaded!')

username = api.whoami()['name']
print(f'\nAll uploaded! Your models:')
print(f'  BART: https://huggingface.co/{username}/meeting-bart-samsum')
print(f'  T5:   https://huggingface.co/{username}/meeting-flan-t5')
print(f'  Data: https://huggingface.co/datasets/{username}/meeting-data')
print(f'\nOn Colab, just load with: BartForConditionalGeneration.from_pretrained("{username}/meeting-bart-samsum")')

In [ ]:
import subprocess
# Delete the big full zip, create smaller one
subprocess.run(['rm', '-f', '/kaggle/working/meeting_intelligence_export.zip'], capture_output=True)
subprocess.run(['rm', '-f', '/kaggle/working/1_bart_samsum.zip'], capture_output=True)
subprocess.run(['rm', '-f', '/kaggle/working/2_flan_t5.zip'], capture_output=True)

# Create ONE small zip â€” BART model only (1.4GB)
subprocess.run(['zip', '-r', '/kaggle/working/bart_for_gradio.zip', 
                '/kaggle/working/meeting_intelligence_export/bart_samsum_model',
                '/kaggle/working/meeting_intelligence_export/data'], capture_output=True)

import os
size = os.path.getsize('/kaggle/working/bart_for_gradio.zip') / (1024**2)
print(f'bart_for_gradio.zip: {size:.0f} MB')
print('Now click Save Version â†’ Quick Save â†’ then check Output tab')

In [ ]:
import subprocess, os

# Create zips from the ACTUAL model locations
print('Creating zip files...')

subprocess.run(['zip', '-r', '/kaggle/working/bart_model.zip', 
                '/kaggle/working/meeting_intelligence/models/model_bart_samsum_best'], capture_output=True)
print('1. bart_model.zip done')

subprocess.run(['zip', '-r', '/kaggle/working/t5_model.zip', 
                '/kaggle/working/meeting_intelligence/models/model_flan_t5_best'], capture_output=True)
print('2. t5_model.zip done')

subprocess.run(['zip', '-r', '/kaggle/working/data.zip', 
                '/kaggle/working/meeting_intelligence/data'], capture_output=True)
print('3. data.zip done')

# Check sizes
for f in ['bart_model.zip', 't5_model.zip', 'data.zip']:
    p = f'/kaggle/working/{f}'
    if os.path.exists(p):
        print(f'  {f}: {os.path.getsize(p)/(1024**2):.0f} MB')
    else:
        print(f'  {f}: NOT FOUND')

print('\nNow click: Save Version â†’ Quick Save')
print('Then go to notebook page â†’ Output tab â†’ download the zips')